<a href="https://colab.research.google.com/github/sarahliu-mma/Quantitative-Trading-Pipeline_FIAM-Hackathon-2026/blob/main/notebooks/02_data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Engineering


### Build stock panel data
Output:
*   Identifier: primary key(permno, eom)
*   Features: 147 JKP features
*   Target: 'ret_exc_lead1m'
*   Auxiliary columns

### Sanity Check
*   No duplicated primary key







#### 1. Setting & Load the Data

In [61]:
import numpy as np
import pandas as pd

In [62]:
dt = pd.read_parquet("/content/drive/MyDrive/FIAM Hackathon/chars_final_with_names.parquet")
jkp_features = pd.read_csv("/content/drive/MyDrive/FIAM Hackathon/factor_char_list.csv")

In [63]:
dt.head()

,id,permno,permco,gvkey,iid,excntry,exch_main,common,primary_sec,bidask,...,qmj_prof,qmj_growth,qmj_safety,qmj,me_lag1,ticker,company_name,ticker_name_reference_date,ticker_name_source,ticker_name_status
0,10001,10001,7953,012994,01,USA,1,1,1,0,...,-0.793438,-1.174184,-0.554461,-1.339223,115.57776,EGAS,GAS NATURAL INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
1,10025,10025,7975,011903,01,USA,1,1,1,0,...,-0.659718,-1.648672,-1.076800,-1.539792,295.46015,AEPI,A E P INDUSTRIES INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
2,10026,10026,7976,012825,01,USA,1,1,1,0,...,1.291849,-1.083642,1.560033,1.060720,2031.60606,JJSF,J & J SNACK FOODS CORP,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
3,10028,10028,7978,012096,01,USA,1,1,1,0,...,0.288480,1.542084,-0.634534,0.691674,14.91328,DGSE,D G S E COMPANIES INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
4,10032,10032,7980,012945,01,USA,1,1,1,0,...,-0.891624,-0.555288,0.647570,-0.687090,1384.86205,PLXS,PLEXUS CORP,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month


2. Build panel data

In [64]:
print(list(dt.columns.difference(jkp_features['variable'])))

['adjfct', 'bidask', 'common', 'comp_exchg', 'comp_tpci', 'company_name', 'conditionaltype', 'crsp_exchcd', 'crsp_nyse', 'crsp_shrcd', 'curcd', 'date', 'dolvol', 'eom', 'exch_main', 'exchcd', 'excntry', 'ff49', 'fx', 'gics', 'gvkey', 'id', 'iid', 'me', 'me_company', 'me_lag1', 'naics', 'obs_main', 'permco', 'permno', 'prc_high', 'prc_local', 'prc_low', 'primary_sec', 'primaryexch', 'ret', 'ret_exc', 'ret_exc_lead1m', 'ret_exc_wins', 'ret_lag_dif', 'ret_local', 'shares', 'shrcd', 'sic', 'size_grp', 'source_crsp', 'ticker', 'ticker_name_reference_date', 'ticker_name_source', 'ticker_name_status', 'tvol']


In [65]:
col_to_drop = ['id','permco','iid','comp_exchg','crsp_exchcd','exchcd','primaryexch','crsp_shrcd','shrcd','comp_tpci','conditionaltype'
,'excntry','curcd','fx','ret_local','ret','ret_lag_dif','prc_high','prc_low','adjfct','me_company','shares','tvol','sic','naics','gics',
                    'source_crsp','ticker_name_reference_date','ticker_name_source','ticker_name_status'
]

In [66]:
panel_dt = dt.drop(columns= col_to_drop)

In [67]:
panel_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529082 entries, 0 to 529081
Columns: 168 entries, permno to company_name
dtypes: float64(154), int32(6), int64(2), object(6)
memory usage: 666.0+ MB


In [68]:
panel_dt.select_dtypes('object').columns

Index(['gvkey', 'date', 'eom', 'size_grp', 'ticker', 'company_name'], dtype='object')

In [69]:
panel_dt['eom'] = pd.to_datetime(panel_dt['eom'])

In [70]:
float64_col = panel_dt.select_dtypes('float64').columns

In [71]:
panel_dt[float64_col] = panel_dt[float64_col].astype('float32')

In [72]:
panel_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529082 entries, 0 to 529081
Columns: 168 entries, permno to company_name
dtypes: datetime64[ns](1), float32(154), int32(6), int64(2), object(5)
memory usage: 355.2+ MB


In [73]:
panel_dt.duplicated(['permno','eom']).sum()

np.int64(0)

In [74]:
panel_dt.to_parquet('/content/drive/MyDrive/FIAM Hackathon/panel_dt.parquet',index= False)